# B1 · California housing: baseline, linear model and logarithmic target

**Type**: Implement with scikit-learn · **Level**: ●●○ independent · **Time**: about 35 minutes · **core path ★**

## Objective

Build a linear model for house prices in California, evaluate it against a baseline, and find out that
transforming the target into a logarithm changes which model comes out best **depending on the metric**.

## Dataset

California housing (`ds.california()`): 20,640 districts, 8 numeric features. The target `y` is the median
house value in hundreds of thousands of dollars. On first use it is downloaded and kept in `data/`.
The target is **capped at 5.0**: all the most expensive houses appear with the same value.

## Educational purpose

Practice the complete scikit-learn workflow (split, baseline, fit, metrics) and learn to use
`TransformedTargetRegressor`. The most important point is that "best" is not a property of the model: it depends
on what you measure.

## Task

1. Split into training and test (80/20, `random_state=0`) and call the pieces `X_tr, X_te, y_tr, y_te`.
2. Train a baseline `DummyRegressor(strategy="mean")` and save in `rmse_base` its RMSE on the test set.
3. Train `linear = LinearRegression()` and compute `rmse_lin`, `mae_lin`, `mape_lin` (mean percentage error) and `r2_lin` on the test set. The predictions are called `pred_lin`.
4. Build `log_linear`, a `TransformedTargetRegressor` that trains the same linear regression on the logarithm of the target and brings the predictions back to the original unit. Compute `rmse_log`, `mae_log`, `mape_log` **in the original units** (predictions in `pred_log`).
5. Assign to `best_by_rmse` and to `best_by_mape` the string `"linear"` or `"log"`, based on the numbers you obtained.
6. The price is capped at 5.0: build the mask `capped = (y_te >= 5.0)`, compute `capped_share` (the fraction of capped houses in the test set) and compare the RMSE of the linear model on the capped houses (`rmse_capped`) with the one on the others (`rmse_others`).

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.compose import TransformedTargetRegressor
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (mean_absolute_error, mean_absolute_percentage_error,
                             mean_squared_error, r2_score)
from sklearn.model_selection import train_test_split

X, y = ds.california()
print(X.shape, "| min/max price:", y.min(), y.max())

## Your work

In [ ]:
X_tr, X_te, y_tr, y_te = ...

In [ ]:
base = DummyRegressor(strategy="mean").fit(X_tr, y_tr)
rmse_base = ...

In [ ]:
linear = ...
pred_lin = ...
rmse_lin, mae_lin, mape_lin, r2_lin = ..., ..., ..., ...

In [ ]:
log_linear = TransformedTargetRegressor(regressor=..., func=..., inverse_func=...)
log_linear.fit(X_tr, y_tr)
pred_log = ...
rmse_log, mae_log, mape_log = ..., ..., ...

In [ ]:
print(f"linear : RMSE {rmse_lin:.3f} | MAE {mae_lin:.3f} | MAPE {mape_lin:.3f}")
print(f"log    : RMSE {rmse_log:.3f} | MAE {mae_log:.3f} | MAPE {mape_log:.3f}")
best_by_rmse = ...
best_by_mape = ...

In [ ]:
capped = ...
capped_share = ...
rmse_capped = ...
rmse_others = ...

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`TransformedTargetRegressor(regressor=LinearRegression(), func=np.log, inverse_func=np.exp)`. The MAPE is `mean_absolute_percentage_error(y_te, pred)`. To apply the mask: `mean_squared_error(y_te[capped], pred_lin[capped], squared=False)`.'):
    check(len(X_te) == round(0.2 * len(X)), "the split is 80/20", "X_te should have 20% of the rows")
    check(rmse_lin < rmse_base, "the linear model beats the baseline",
          "the linear model should have a lower RMSE than the baseline")
    check(close(rmse_lin, mean_squared_error(y_te, linear.predict(X_te), squared=False)),
          "rmse_lin is computed correctly", "rmse_lin does not match the predictions of `linear`")
    check(close(rmse_log, mean_squared_error(y_te, log_linear.predict(X_te), squared=False)),
          "rmse_log is computed in the original units",
          "rmse_log does not match log_linear.predict(X_te): the predictions must already be in the original unit")
    check(log_linear.predict(X_te).min() > 0, "TransformedTargetRegressor returns positive prices",
          "with func=np.log and inverse_func=np.exp the predictions must be positive")
    check(best_by_rmse == ("linear" if rmse_lin < rmse_log else "log"), "choice consistent with the RMSE",
          "best_by_rmse is not consistent with rmse_lin and rmse_log")
    check(best_by_mape == ("linear" if mape_lin < mape_log else "log"), "choice consistent with the MAPE",
          "best_by_mape is not consistent with mape_lin and mape_log")
    check(abs(capped_share - (y_te >= 5.0).mean()) < 1e-9, "correct share of capped houses",
          "capped_share must be the mean of the mask (y_te >= 5.0)")
    if not ds.CALIFORNIA_SYNTHETIC:
        check(best_by_rmse != best_by_mape,
              "the two criteria choose DIFFERENT models: that is the point of the exercise",
              "with RMSE and MAPE you should find two different winners: review the calculations, all in the original unit")
        check(rmse_capped > 1.5 * rmse_others, "on the capped houses the model is much more wrong",
              "the RMSE on the capped houses should be much higher")
done()

---
The commented solution is in `solutions/B_with_sklearn/B1_california_baseline_and_log_target.ipynb`. Open it after you pass the check, or after you have really tried.